In [1]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [2]:
documents[2]

{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 'section': 'General course-related questions',
 'question': 'Course - Can I still join the course after the start date?',
 'course': 'data-engineering-zoomcamp'}

In [4]:
!pip3 install minsearch --quiet

import minsearch

index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [8]:
import os
from dotenv import load_dotenv
import os

# Load variables from the `.env` file into the environment
load_dotenv()

# Access the key
api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=api_key)

def search(query):
    boost = {'question': 3.0, 'section': 0.5}
    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )
    return results

def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [9]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

rag('how do I run kafka?')

"To run Kafka, you can follow these instructions based on the specific setup you're using. \n\nFor Java Kafka, in your project directory, you need to execute the following command in the terminal:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```\n\nIf you are working with Python and running a Kafka producer, ensure you have created a virtual environment and installed the necessary packages as follows:\n\n1. Create a virtual environment and install the required packages:\n   ```bash\n   python -m venv env\n   source env/bin/activate\n   pip install -r ../requirements.txt\n   ```\n\n2. Activate the virtual environment (run this command each time you need it):\n   ```bash\n   source env/bin/activate\n   ```\n\n3. Deactivate it when you're done:\n   ```bash\n   deactivate\n   ```\n\nMake sure Docker images are up and running if your setup requires Docker."

In [10]:
rag('the course has already started, can I still enroll?')

"Yes, you can still enroll in the course even though it has already started. You are eligible to submit the homeworks, but keep in mind that there will be deadlines for turning in the final projects, so it's advisable not to leave everything to the last minute."

## RAG with Vector Search

In [11]:
from qdrant_client import QdrantClient, models

qd_client = QdrantClient("http://localhost:6333")
EMBEDDING_DIMENSIONALITY = 512
model_handle = "jinaai/jina-embeddings-v2-small-en"
collection_name = "zoomcamp-faq"

qd_client.delete_collection(collection_name=collection_name)

qd_client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=EMBEDDING_DIMENSIONALITY,
        distance=models.Distance.COSINE
    )
)

True

In [12]:
qd_client.create_payload_index(
    collection_name=collection_name,
    field_name="course",
    field_schema="keyword"
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [13]:
points = []

for i, doc in enumerate(documents):
    text = doc['question'] + ' ' + doc['text']
    vector = models.Document(text=text, model=model_handle)
    point = models.PointStruct(
        id=i,
        vector=vector,
        payload=doc
    )
    points.append(point)

qd_client.upsert(
    collection_name=collection_name,
    points=points
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [14]:
def vector_search(question):
    print('vector_search is used')
    course = 'data-engineering-zoomcamp'
    query_points = qd_client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=question,
            model=model_handle 
        ),
        query_filter=models.Filter( 
            must=[
                models.FieldCondition(
                    key="course",
                    match=models.MatchValue(value=course)
                )
            ]
        ),
        limit=5,
        with_payload=True
    )
    results = []
    for point in query_points.points:
        results.append(point.payload)
    return results
    
def rag(query):
    search_results = vector_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer
    
rag('how do I run kafka?')

vector_search is used


"To run Kafka, you need to do the following steps based on the context provided:\n\n1. **Ensure your Kafka broker is running**: Use the command `docker ps` to confirm that your Kafka broker's Docker container is active. If it's not running, navigate to the folder containing the docker-compose YAML file and execute the command `docker compose up -d` to start all the instances.\n\n2. **Run the Kafka producer or consumer**: In your project directory, execute the following command to run your Kafka producer or consumer (replace `<jar_name>` with the appropriate jar file name):\n   ```bash\n   java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n   ```\n\n3. **Check configurations**: Ensure that the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` in your Java scripts (like `JsonProducer.java` or `JsonConsumer.java`) points to the correct Kafka server URL. Also, verify that the Kafka cluster key and secrets in the `src/main/java/org/example/Secrets.java` 

In [15]:
question = 'I just discovered the course. Can I still join it?'
rag(question)

vector_search is used


'Yes, you can still join the course even if it has already started. You are eligible to submit homework assignments. However, be mindful that there are deadlines for turning in the final projects, so it’s best not to leave everything until the last minute.'